# 🔋 LIF, LIF-AT, and AdEx Neuron Models

Three leaky integrate-and-fire variants:

**LIF:** $$\tau_m \dot{v} = -(v - v_{rest}) + I$$

**LIF-AT:** adds adaptive threshold $$\theta$$ that increases after each spike

**AdEx (Adaptive Exponential):** adds exponential term and adaptation current w

$$\tau_m \dot{v} = -(v - v_{rest}) + \Delta_T \exp\left(\frac{v - v_T}{\Delta_T}\right) - w + I$$
$$\tau_w \dot{w} = a(v - v_{rest}) - w$$


## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    import taas_checkmate as ne
except ImportError:
    import subprocess
    subprocess.run(["maturin", "develop", "--release", "-q"], check=True)
    import taas_checkmate as ne

plt.style.use("dark_background")
N = 512
N_PAD = (N + 7) // 8 * 8
DT = 0.5

print("✅ Setup complete")


### 2a. LIF Simulation

In [ ]:
# LIF parameters
v = np.full(N_PAD, -65.0, dtype=np.float32)
I = np.full(N_PAD, 8.0, dtype=np.float32)
spikes = np.zeros(N_PAD, dtype=np.int32)
tau_m = np.full(N_PAD, 20.0, dtype=np.float32)
v_rest = np.full(N_PAD, -65.0, dtype=np.float32)
v_thresh = np.full(N_PAD, -50.0, dtype=np.float32)
v_reset = np.full(N_PAD, -70.0, dtype=np.float32)

STEPS = 1000
spike_buffer_lif = np.zeros((STEPS, N), dtype=np.int32)

for t in range(STEPS):
    ne.step_lif(v, I, spikes, DT, tau_m, v_rest, v_thresh, v_reset)
    spike_buffer_lif[t] = spikes[:N]

print(f"LIF firing rate: {spike_buffer_lif.mean() * 1000 / DT:.1f} Hz")


### 2b. LIF-AT (Adaptive Threshold)

In [ ]:
# LIF-AT with stateful wrapper
lif_at = ne.PyLifNeuron(N_PAD, dt=DT, tau_m=20.0, v_rest=-65.0,
                         v_thresh=-50.0, v_reset=-70.0,
                         tau_theta=50.0, beta=0.5)

I = np.full(N_PAD, 8.0, dtype=np.float32)
spikes = np.zeros(N_PAD, dtype=np.int32)
v_at = np.zeros(N_PAD, dtype=np.float32)
theta_at = np.zeros(N_PAD, dtype=np.float32)
v_history = np.zeros((STEPS, N), dtype=np.float32)
theta_history = np.zeros((STEPS, N), dtype=np.float32)
spike_buffer_at = np.zeros((STEPS, N), dtype=np.int32)

for t in range(STEPS):
    lif_at.step(I, spikes)
    spike_buffer_at[t] = spikes[:N]
    lif_at.copy_v(v_at)
    v_history[t] = v_at[:N]
    lif_at.copy_theta(theta_at)
    theta_history[t] = theta_at[:N]

print(f"LIF-AT firing rate: {spike_buffer_at.mean() * 1000 / DT:.1f} Hz")
print(f"  Mean theta: {theta_history.mean():.2f}")


### 2c. AdEx (Adaptive Exponential)

In [ ]:
v = np.full(N_PAD, -65.0, dtype=np.float32)
u = np.full(N_PAD, 0.0, dtype=np.float32)
I = np.full(N_PAD, 12.0, dtype=np.float32)
spikes = np.zeros(N_PAD, dtype=np.int32)
tau_m = np.full(N_PAD, 20.0, dtype=np.float32)
tau_w = np.full(N_PAD, 100.0, dtype=np.float32)
v_rest = np.full(N_PAD, -65.0, dtype=np.float32)
v_thresh = np.full(N_PAD, -50.0, dtype=np.float32)
v_reset = np.full(N_PAD, -70.0, dtype=np.float32)
delta_t = np.full(N_PAD, 2.0, dtype=np.float32)
a = np.full(N_PAD, 0.02, dtype=np.float32)
b = np.full(N_PAD, 0.02, dtype=np.float32)

spike_buffer_adex = np.zeros((STEPS, N), dtype=np.int32)

for t in range(STEPS):
    ne.step_adex(v, u, I, spikes, DT, tau_m, tau_w, v_rest,
                 v_thresh, v_reset, delta_t, a, b)
    spike_buffer_adex[t] = spikes[:N]

print(f"AdEx firing rate: {spike_buffer_adex.mean() * 1000 / DT:.1f} Hz")


## Comparison Raster

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

for ax, buf, title, color in zip(
    axes,
    [spike_buffer_lif, spike_buffer_at, spike_buffer_adex],
    ["LIF", "LIF-AT (Adaptive Threshold)", "AdEx (Adaptive Exponential)"],
    ["#00ff88", "#ff6600", "#3399ff"],
):
    t, n = np.where(buf > 0)
    ax.scatter(t * DT, n, s=0.3, c=color, alpha=0.4, rasterized=True)
    ax.set_ylabel("Neuron")
    ax.set_title(title, color=color)
    ax.set_ylim(0, N)
    ax.grid(alpha=0.1)

axes[-1].set_xlabel("Time (ms)")
plt.tight_layout()
plt.show()


## Adaptive Threshold Dynamics

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Sample neuron
idx = 42
ax1.plot(np.arange(STEPS) * DT, v_history[:, idx],
         color="#00ff88", linewidth=0.8, label="V")
ax1.plot(np.arange(STEPS) * DT, theta_history[:, idx],
         color="#ff6600", linewidth=0.8, label="θ (threshold)")
ax1.axhline(-50, color="gray", linestyle="--", alpha=0.3,
            label="Base threshold")
ax1.set_xlabel("Time (ms)")
ax1.set_ylabel("Membrane Potential")
ax1.set_title(f"LIF-AT: V and θ for Neuron {idx}")
ax1.legend()
ax1.grid(alpha=0.1)

# Theta distribution
mean_theta = theta_history.mean(axis=0)
ax2.hist(mean_theta[:N], bins=30, color="#ff6600", alpha=0.7, edgecolor="none")
ax2.set_xlabel("Mean θ")
ax2.set_ylabel("Count")
ax2.set_title("Adaptive Threshold Distribution")
ax2.grid(alpha=0.1)

plt.tight_layout()
plt.show()

print(f"θ range: [{mean_theta[:N].min():.2f}, {mean_theta[:N].max():.2f}]")
